# 01 — Exploration

First look at the six raw ACNH source CSVs (`data/source/`) before any
cleaning happens. Goal: understand row counts, column shapes, and the
kinds of dirty data the transformation layer (`sql/04`) needs to handle
(inconsistent casing, blank prices, `NFS` buy values, duplicated rows).

In [1]:
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd().parent
SOURCE_DIR = BASE_DIR / "data" / "source"

TABLES = ["fish", "insects", "fossils", "villagers", "housewares", "recipes"]

dataframes = {
    table: pd.read_csv(SOURCE_DIR / f"{table}.csv", dtype=str, encoding="utf-8-sig")
    for table in TABLES
}

{table: df.shape for table, df in dataframes.items()}

{'fish': (80, 41),
 'insects': (80, 38),
 'fossils': (73, 14),
 'villagers': (391, 17),
 'housewares': (3275, 32),
 'recipes': (595, 24)}

## Row and column counts

In [2]:
summary = pd.DataFrame(
    {
        "rows": {t: len(df) for t, df in dataframes.items()},
        "columns": {t: df.shape[1] for t, df in dataframes.items()},
    }
)
summary

,rows,columns
fish,80,41
insects,80,38
fossils,73,14
villagers,391,17
housewares,3275,32
recipes,595,24


## Missing values per table

The source CSVs use empty strings for missing data, not `NaN`, so this
counts empty/whitespace-only cells explicitly.

In [3]:
def count_blanks(df: pd.DataFrame) -> int:
    return int((df.fillna("").apply(lambda col: col.str.strip()) == "").sum().sum())


{table: count_blanks(df) for table, df in dataframes.items()}

{'fish': 866,
 'insects': 1018,
 'fossils': 0,
 'villagers': 0,
 'housewares': 18801,
 'recipes': 5844}

## `Sell` price distribution

The Monthly Bell Guide dashboard ranks items by `Sell`, so this is the
column that matters most for the business question. `fossils`, `housewares`,
and `recipes` also use the literal string `"NFS"` (Not For Sale) in `Buy`,
which `sql/04`'s transformation procedures turn into `NULL`.

In [4]:
for table in ["fish", "insects", "fossils", "housewares", "recipes"]:
    sell = pd.to_numeric(dataframes[table]["Sell"], errors="coerce")
    print(table)
    print(sell.describe())
    print()

fish
count       80.000000
mean      3745.000000
std       4517.151561
min        100.000000
25%        500.000000
50%       1500.000000
75%       5000.000000
max      15000.000000
Name: Sell, dtype: float64

insects
count       80.000000
mean      2220.500000
std       3209.388877
min         10.000000
25%        237.500000
50%        600.000000
75%       2625.000000
max      12000.000000
Name: Sell, dtype: float64

fossils
count      73.000000
mean     3563.013699
std      1402.924213
min      1000.000000
25%      2500.000000
50%      4000.000000
75%      4500.000000
max      6000.000000
Name: Sell, dtype: float64

housewares
count      3275.000000
mean       4011.301985
std       13823.831987
min          20.000000
25%         575.000000
50%        1250.000000
75%        2400.000000
max      250000.000000
Name: Sell, dtype: float64

recipes
count    595.0
mean     200.0
std        0.0
min      200.0
25%      200.0
50%      200.0
75%      200.0
max      200.0
Name: Sell, dtype: float

## Name casing

Source names are already consistent (Title Case). The dirty-data casing
issues (`lower`, `UPPER`, extra whitespace) only show up in the demo
batches 2/3 built by `scripts/split_batches.py`, not in `data/source/`.

In [5]:
dataframes["fish"]["Name"].head(10)

0            anchovy
1          angelfish
2           arapaima
3            arowana
4    barred knifejaw
5          barreleye
6              betta
7         bitterling
8         black bass
9           blowfish
Name: Name, dtype: str